https://tianchi.aliyun.com/notebook/144453

## Task 4  特征工程

召回后，分数不一定能够直接比较，比如多路召回后UserCF、ItemCF提供的文章未必有相同的点击概率。我们需要挖掘更多的特征，训练一个排序模型，让模型综合使用召回分数、用户兴趣特征、文章属性来学习哪些候选更可能被点击：

- 基于日志数据集，获得用户最后一次点击操作和用户的历史点击

- 基于用户的历史行为制作特征，用到用户历史点击表、召回列表、文章信息表、embedding向量

- 制作标签，形成最后的监督学习数据集

**问题：训练排序模型，哪些候选是正样本、负样本**

- rank_hist: 排序训练模型允许看到的用户历史

- rank_targets:排序模型需要尝试的用户下一次真实点击

In [ ]:

# 前置准备
# ============================================================
# 1. 导入依赖库
# ============================================================

# Python 标准库
import sys
import os
import gc
import pickle
import logging
import time

# 数据处理
import numpy as np
import pandas as pd

# 进度显示
from tqdm import tqdm

# 特征工程
from sklearn.preprocessing import MinMaxScaler
from gensim.models import Word2Vec

# ============================================================
# 2. 导入项目公共模块
# ============================================================

# Notebook 位于 Task-4/，公共模块位于上一级 metric/
sys.path.insert(0, os.path.abspath(".."))

from metric.data import (
    prepare_evaluation,
    split_last_click
)

from metric.evaluation import (
    evaluate_predictions,
    evaluate_csv,
    save_predictions
)

# ============================================================
# 3. 读取原始训练数据
# ============================================================

train_clicks = pd.read_csv(
    "../data/train_click_log.csv"
)

# ============================================================
# 4. 第一层：固定离线评估划分
# ============================================================

fit_clicks, query_clicks, targets = prepare_evaluation(
    train_clicks
)

# ============================================================
# 5. 第二层：排序模型监督训练数据划分
# ============================================================

rank_hist, rank_targets = split_last_click(
    fit_clicks
)

# ============================================================
# 6. 数据划分检查
# ============================================================

click_cols = [
    "user_id",
    "click_article_id",
    "click_timestamp"
]

# 固定验证用户数量
assert targets["user_id"].nunique() == 50_000

# 验证用户必须具有可见历史
assert targets["user_id"].isin(
    query_clicks["user_id"]
).all()

# 外层验证标签不得出现在拟合数据中
assert targets.merge(
    fit_clicks,
    on=click_cols
).empty

# 内层排序训练标签不得出现在排序训练历史中
assert rank_targets.merge(
    rank_hist,
    on=click_cols
).empty

# 内层划分应完整覆盖 fit_clicks
assert len(rank_hist) + len(rank_targets) == len(fit_clicks)

print("fit:", fit_clicks.shape)
print("query:", query_clicks.shape)
print("evaluation targets:", targets.shape)
print("ranking history:", rank_hist.shape)
print("ranking targets:", rank_targets.shape)


fit: (1062623, 3)
query: (227284, 3)
evaluation targets: (50000, 3)
ranking history: (882966, 3)
ranking targets: (179657, 3)


In [4]:

# 节省内存的一个函数
# 减少内存
def reduce_mem(df):
    starttime = time.time()
    numerics = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']
    start_mem = df.memory_usage().sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtypes
        if col_type in numerics:
            c_min = df[col].min()
            c_max = df[col].max()
            if pd.isnull(c_min) or pd.isnull(c_max):
                continue
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
    end_mem = df.memory_usage().sum() / 1024**2
    print('-- Mem. usage decreased to {:5.2f} Mb ({:.1f}% reduction),time spend:{:2.2f} min'.format(end_mem,
                                                                                                           100*(start_mem-end_mem)/start_mem,
                                                                                                           (time.time()-starttime)/60))
    return df


In [6]:
data_path = '../data/'
save_path = './results/'
os.makedirs(save_path, exist_ok=True)

## 数据读取

In [10]:
# ============================================================
# 训练和验证集的划分
# ============================================================

# all_click_df 指的是训练集
# sample_user_nums 采样作为验证集的用户数量
def trn_val_split(all_click_df, sample_user_nums):

    # 复用公共离线评估划分，替代原文的随机抽样
    click_trn, click_val, val_ans = prepare_evaluation(
        all_click_df,
        n_users=sample_user_nums
    )

    return click_trn, click_val, val_ans

In [11]:

# ============================================================
# 获取历史点击和最后一次点击
# ============================================================

def get_hist_and_last_click(all_click):

    # 复用公共末次点击划分，避免单击用户标签泄漏
    click_hist_df, click_last_df = split_last_click(all_click)

    return click_hist_df, click_last_df


In [ ]:
# ============================================================
# 读取训练、验证及测试集
# ============================================================
def get_trn_val_tst_data(data_path, offline=True):
    if offline:
        click_trn_data = pd.read_csv(data_path + 'train_click_log.csv')  # 训练集用户点击日志
        click_trn_data = reduce_mem(click_trn_data)

        # 使用已确认的公共离线划分，固定50000名验证用户
        click_trn, click_val, val_ans = trn_val_split(
            click_trn_data, sample_user_nums=50000
        )
    else:
        click_trn = pd.read_csv(data_path + 'train_click_log.csv')
        click_trn = reduce_mem(click_trn)
        click_val = None
        val_ans = None

    click_tst = pd.read_csv(data_path + 'testA_click_log.csv')

    return click_trn, click_val, click_tst, val_ans


In [13]:

# ============================================================
# 读取召回列表
# ============================================================

# Task3 已经生成的召回结果目录
recall_path = '../Task-3/results/'


# 返回多路召回列表或者单路召回
def get_recall_list(save_path, single_recall_model=None, multi_recall=False):
    if multi_recall:
        return pickle.load(open(save_path + 'final_recall_items_dict.pkl', 'rb'))

    if single_recall_model == 'i2i_itemcf':
        return pickle.load(open(save_path + 'itemcf_recall_dict.pkl', 'rb'))

    elif single_recall_model == 'i2i_emb_itemcf':
        return pickle.load(open(save_path + 'embedding_sim_item_recall.pkl', 'rb'))

    elif single_recall_model == 'user_cf':
        return pickle.load(open(save_path + 'youtubednn_usercf_recall.pkl', 'rb'))

    elif single_recall_model == 'youtubednn':
        return pickle.load(open(save_path + 'youtube_u2i_dict.pkl', 'rb'))

    else:
        raise ValueError("未指定有效的单路召回模型或多路召回模式")


## 读取各种Embedding

### Word2Vec训练以及gensim使用

Word2Vec主要思想是：一个词的上下文可以很好的表达出词的语义。通过无监督学习产生词向量的方式。word2vec中有两个非常经典的模型：skip-gram和cbow

- skip-gram：已知中心词预测周围词。
- cbow：已知周围词预测中心词。

在使用gensim训练word2vec的时候，有几个比较重要的参数
- size: 表示词向量的维度。
- window：决定了目标词会与多远距离的上下文产生关系。
- sg: 如果是0，则是CBOW模型，是1则是Skip-Gram模型。
- workers: 表示训练时候的线程数量
- min_count: 设置最小的
- iter: 训练时遍历整个数据集的次数

In [14]:

# ============================================================
# 读取各种 Embedding
# ============================================================
w2v_save_path = './results/offline_rank/'


# Word2Vec 训练文章 Embedding
def trian_item_word2vec(click_df, embed_size=16,
                        save_name='item_w2v_emb.pkl',
                        split_char=' ', w2v_save_path=None):

    click_df = click_df.sort_values('click_timestamp').copy()

    # 只有转换成字符串才可以作为 Word2Vec 的词进行训练
    click_df['click_article_id'] = click_df['click_article_id'].astype(str)

    # 转换成句子的形式
    docs = click_df.groupby(['user_id'])['click_article_id'].apply(
        lambda x: list(x)
    ).reset_index()

    docs = docs['click_article_id'].values.tolist()

    # 设置日志信息
    logging.basicConfig(
        format='%(asctime)s:%(levelname)s:%(message)s',
        level=logging.INFO
    )

    # Skip-Gram、window=5、训练1轮的设置
    w2v = Word2Vec(
        docs,
        vector_size=embed_size,
        sg=1,
        window=5,
        seed=2020,
        workers=24,
        min_count=1,
        epochs=1
    )

    # 保存成字典形式，文章ID统一使用整数
    item_w2v_emb_dict = {
        int(k): w2v.wv[k]
        for k in click_df['click_article_id'].unique()
    }

    output_path = (
        w2v_save_path if w2v_save_path is not None
        else save_path
    )
    os.makedirs(output_path, exist_ok=True)

    with open(output_path + save_name, 'wb') as f:
        pickle.dump(item_w2v_emb_dict, f)

    return item_w2v_emb_dict


# 读取已保存的各种 Embedding
def get_embedding(save_path, all_click_df, w2v_save_path=None):

    # 文章内容 Embedding
    if os.path.exists(save_path + 'item_content_emb.pkl'):
        item_content_emb_dict = pickle.load(
            open(save_path + 'item_content_emb.pkl', 'rb')
        )
    else:
        print('item_content_emb.pkl 文件不存在...')
        item_content_emb_dict = None

    # Word2Vec Embedding
    w2v_path = (
        w2v_save_path if w2v_save_path is not None
        else save_path
    )

    if os.path.exists(w2v_path + 'item_w2v_emb.pkl'):
        item_w2v_emb_dict = pickle.load(
            open(w2v_path + 'item_w2v_emb.pkl', 'rb')
        )
    else:
        item_w2v_emb_dict = trian_item_word2vec(
            all_click_df,
            w2v_save_path=w2v_path
        )

    # YouTubeDNN 文章 Embedding
    if os.path.exists(save_path + 'item_youtube_emb.pkl'):
        item_youtube_emb_dict = pickle.load(
            open(save_path + 'item_youtube_emb.pkl', 'rb')
        )
    else:
        print('item_youtube_emb.pkl 文件不存在...')
        item_youtube_emb_dict = None

    # YouTubeDNN 用户 Embedding
    if os.path.exists(save_path + 'user_youtube_emb.pkl'):
        user_youtube_emb_dict = pickle.load(
            open(save_path + 'user_youtube_emb.pkl', 'rb')
        )
    else:
        print('user_youtube_emb.pkl 文件不存在...')
        user_youtube_emb_dict = None

    return (
        item_content_emb_dict,
        item_w2v_emb_dict,
        item_youtube_emb_dict,
        user_youtube_emb_dict
    )


In [15]:

# 读取文章信息
def get_article_info_df():
    article_info_df = pd.read_csv(data_path + 'articles.csv')
    article_info_df = reduce_mem(article_info_df)

    return article_info_df


In [ ]:

# ============================================================
# 读取数据
# ============================================================

# 使用固定的离线验证集，offline=True
click_trn, click_val, click_tst, val_ans = get_trn_val_tst_data(
    data_path, offline=True
)

# 获取训练集历史点击和最后一次点击
click_trn_hist, click_trn_last = get_hist_and_last_click(click_trn)

if click_val is not None:
    click_val_hist, click_val_last = click_val, val_ans
else:
    click_val_hist, click_val_last = None, None

# 测试集的全部可见点击就是其历史点击
click_tst_hist = click_tst

# 读取文章信息
article_info_df = get_article_info_df()

print('click_trn:', click_trn.shape)
print('click_val:', click_val.shape)
print('click_tst:', click_tst.shape)
print('val_ans:', val_ans.shape)

print('click_trn_hist:', click_trn_hist.shape)
print('click_trn_last:', click_trn_last.shape)
print('article_info_df:', article_info_df.shape)


-- Mem. usage decreased to 23.34 Mb (69.4% reduction),time spend:0.00 min
-- Mem. usage decreased to  5.56 Mb (50.0% reduction),time spend:0.00 min
click_trn: (1062623, 3)
click_val: (227284, 3)
click_tst: (518010, 9)
val_ans: (50000, 3)
click_trn_hist: (882966, 3)
click_trn_last: (179657, 3)
article_info_df: (364047, 4)


## 对训练数据做负采样

In [17]:

# 将召回列表转换成df的形式
def recall_dict_2_df(recall_list_dict):
    df_row_list = [] # [user, item, score]
    for user, recall_list in tqdm(recall_list_dict.items()):
        for item, score in recall_list:
            df_row_list.append([user, item, score])

    col_names = ['user_id', 'sim_item', 'score']
    recall_list_df = pd.DataFrame(df_row_list, columns=col_names)

    return recall_list_df


In [ ]:

# ============================================================
# 读取召回列表
# ============================================================

# 沿用原文，先使用单路 ItemCF 召回
recall_list_dict = get_recall_list(
    recall_path,
    single_recall_model='i2i_itemcf'
)

# 将召回数据转换成 df
val_recall_list_df = recall_dict_2_df(recall_list_dict)


# ============================================================
# 检查转换结果
# ============================================================

display(val_recall_list_df.head())


100%|██████████| 50000/50000 [00:01<00:00, 25362.12it/s]


召回字典用户数: 50000
转换后的数据维度: (2500000, 3)
有候选文章的用户数: 50000


,user_id,sim_item,score
0,0,36162,0.857716
1,0,30955,0.758154
2,0,234481,0.689305
3,0,180441,0.672240
4,0,50644,0.567381


In [30]:

# 负采样函数，这里可以控制负采样时的比例，这里给了一个默认的值
def neg_sample_recall_data(recall_items_df, sample_rate=0.001):
    pos_data = recall_items_df[recall_items_df['label'] == 1]
    neg_data = recall_items_df[recall_items_df['label'] == 0]

    print('pos_data_num:', len(pos_data),
          'neg_data_num:', len(neg_data),
          'pos/neg:', len(pos_data) / len(neg_data) if len(neg_data) else 0)

    if neg_data.empty:
        return pos_data.reset_index(drop=True)

    # 分组采样函数
    def neg_sample_func(group_df):
        neg_num = len(group_df)
        sample_num = max(int(neg_num * sample_rate), 1)
        sample_num = min(sample_num, 5)
        return group_df.sample(n=sample_num, replace=True)

    # 对用户进行负采样，保证有负候选的用户被覆盖
    # 显式选择全部列，避免 Pandas 新版本丢失分组字段
    neg_data_user_sample = neg_data.groupby(
        'user_id', group_keys=False
    )[neg_data.columns.tolist()].apply(neg_sample_func)

    # 对文章进行负采样，保证有负候选的文章被覆盖
    neg_data_item_sample = neg_data.groupby(
        'sim_item', group_keys=False
    )[neg_data.columns.tolist()].apply(neg_sample_func)

    # 将上述两种情况下的采样数据合并
    neg_data_new = pd.concat([
        neg_data_user_sample,
        neg_data_item_sample
    ])

    # 去除重复的用户与文章配对
    neg_data_new = neg_data_new.sort_values(
        ['user_id', 'score']
    ).drop_duplicates(['user_id', 'sim_item'], keep='last')

    # 将正样本数据合并
    data_new = pd.concat(
        [pos_data, neg_data_new],
        ignore_index=True
    )

    return data_new



In [20]:

# 召回数据打标签
def get_rank_label_df(recall_list_df, label_df, is_test=False):
    # 测试集是没有标签了，为了后面代码统一一些，这里直接给一个负数替代
    if is_test:
        recall_list_df['label'] = -1
        return recall_list_df

    label_df = label_df.rename(columns={'click_article_id': 'sim_item'})
    recall_list_df_ = recall_list_df.merge(
        label_df[['user_id', 'sim_item', 'click_timestamp']],
        how='left',
        on=['user_id', 'sim_item']
    )
    recall_list_df_['label'] = recall_list_df_['click_timestamp'].apply(
        lambda x: 0.0 if np.isnan(x) else 1.0
    )
    del recall_list_df_['click_timestamp']

    return recall_list_df_


In [21]:

# 使用固定验证标签，检查召回数据打标签的结果
val_user_item_label_df = get_rank_label_df(
    val_recall_list_df,
    click_val_last,
    is_test=False
)

print('验证候选总数:', len(val_user_item_label_df))
print('正样本数量:', int(val_user_item_label_df['label'].sum()))
print('固定验证用户数:', click_val_last['user_id'].nunique())

print('\n标签分布:')
print(val_user_item_label_df['label'].value_counts())

assert val_user_item_label_df['label'].isin([0.0, 1.0]).all()
assert val_user_item_label_df.groupby('user_id')['label'].sum().le(1).all()

display(val_user_item_label_df.head())


验证候选总数: 2500000
正样本数量: 30748
固定验证用户数: 50000

标签分布:
label
0.0    2469252
1.0      30748
Name: count, dtype: int64


,user_id,sim_item,score,label
0,0,36162,0.857716,0.0
1,0,30955,0.758154,0.0
2,0,234481,0.689305,0.0
3,0,180441,0.672240,0.0
4,0,50644,0.567381,0.0


In [22]:

# 给训练、验证和测试召回数据打标签，并对训练负样本采样
def get_user_recall_item_label_df(
    click_trn_hist, click_val_hist, click_tst_hist,
    click_trn_last, click_val_last,
    trn_recall_list_df, val_recall_list_df=None,
    tst_recall_list_df=None
):
    # 获取训练数据的召回列表
    # 只保留有真实训练标签的用户，避免单击用户产生全负样本
    trn_users = pd.Index(
        click_trn_hist['user_id'].unique()
    ).intersection(
        click_trn_last['user_id'].unique()
    )

    trn_user_items_df = trn_recall_list_df[
        trn_recall_list_df['user_id'].isin(trn_users)
    ].copy()

    # 训练数据打标签
    trn_user_item_label_df = get_rank_label_df(
        trn_user_items_df, click_trn_last, is_test=False
    )

    # 训练数据负采样
    trn_user_item_label_df = neg_sample_recall_data(
        trn_user_item_label_df
    )

    # 验证数据打标签
    if click_val_hist is not None:
        if val_recall_list_df is None:
            raise ValueError("缺少验证集召回结果")

        val_user_items_df = val_recall_list_df[
            val_recall_list_df['user_id'].isin(
                click_val_hist['user_id'].unique()
            )
        ].copy()

        val_user_item_label_df = get_rank_label_df(
            val_user_items_df, click_val_last, is_test=False
        )

        # 验证集不进行负采样，保留完整候选
    else:
        val_user_item_label_df = None

    # 测试数据不需要负采样，直接对所有召回文章打 -1 标签
    if tst_recall_list_df is not None:
        tst_user_items_df = tst_recall_list_df[
            tst_recall_list_df['user_id'].isin(
                click_tst_hist['user_id'].unique()
            )
        ].copy()

        tst_user_item_label_df = get_rank_label_df(
            tst_user_items_df, None, is_test=True
        )
    else:
        tst_user_item_label_df = None

    return (
        trn_user_item_label_df,
        val_user_item_label_df,
        tst_user_item_label_df
    )


## 重新生成排序训练候选

In [24]:
def itemcf_sim(df, item_created_time_dict):
    """
        文章与文章之间的相似性矩阵计算
        :param df: 数据表
        :item_created_time_dict:  文章创建时间的字典
        return : 文章与文章的相似性矩阵
        
        思路: 基于物品的协同过滤(详细请参考上一期推荐系统基础的组队学习) + 关联规则
    """
    
    user_item_time_dict = get_user_item_time(df)
    
    # 计算物品相似度
    i2i_sim = {}
    item_cnt = defaultdict(int)
    for user, item_time_list in tqdm(user_item_time_dict.items()):
        # 在基于商品的协同过滤优化的时候可以考虑时间因素
        for loc1, (i, i_click_time) in enumerate(item_time_list):
            item_cnt[i] += 1
            i2i_sim.setdefault(i, {})
            for loc2, (j, j_click_time) in enumerate(item_time_list):
                if(i == j):
                    continue
                    
                # 考虑文章的正向顺序点击和反向顺序点击    
                loc_alpha = 1.0 if loc2 > loc1 else 0.7
                # 位置信息权重，其中的参数可以调节
                loc_weight = loc_alpha * (0.9 ** (np.abs(loc2 - loc1) - 1))
                # 点击时间权重，其中的参数可以调节
                click_time_weight = np.exp(0.7 ** np.abs(i_click_time - j_click_time))
                # 两篇文章创建时间的权重，其中的参数可以调节
                created_time_weight = np.exp(0.8 ** np.abs(item_created_time_dict[i] - item_created_time_dict[j]))
                i2i_sim[i].setdefault(j, 0)
                # 考虑多种因素的权重计算最终的文章之间的相似度
                i2i_sim[i][j] += loc_weight * click_time_weight * created_time_weight / math.log(len(item_time_list) + 1)
                
    i2i_sim_ = i2i_sim.copy()
    for i, related_items in i2i_sim.items():
        for j, wij in related_items.items():
            i2i_sim_[i][j] = wij / math.sqrt(item_cnt[i] * item_cnt[j])
    
    # 将得到的相似性矩阵保存到本地
    pickle.dump(i2i_sim_, open(save_path + 'itemcf_i2i_sim.pkl', 'wb'))
    
    return i2i_sim_

In [25]:
def get_user_item_time(click_df):
    click_df = click_df.sort_values('click_timestamp')

    def make_item_time_pair(df):
        return list(zip(
            df['click_article_id'],
            df['click_timestamp']
        ))

    user_item_time_df = (
        click_df
        .groupby('user_id')[['click_article_id', 'click_timestamp']]
        .apply(lambda x: make_item_time_pair(x))
        .reset_index()
        .rename(columns={0: 'item_time_list'})
    )

    user_item_time_dict = dict(zip(
        user_item_time_df['user_id'],
        user_item_time_df['item_time_list']
    ))

    return user_item_time_dict

In [26]:
def item_based_recommend(
    user_id,
    user_item_time_dict,
    i2i_sim,
    sim_item_topk,
    recall_item_num,
    item_topk_click,
    item_created_time_dict,
    emb_i2i_sim
):
    """
    根据用户历史点击文章及 i2i 相似度生成召回结果。

    返回：
        [(article_id, score), ...]
        按 score 从高到低排序。
    """

    # 1. 获取用户历史交互文章
    user_hist_items = user_item_time_dict[user_id]
    user_hist_items_ = {
        item for item, _ in user_hist_items
    }

    item_rank = {}

    # 2. 遍历用户的历史点击
    for loc, (i, click_time) in enumerate(user_hist_items):

        # 取历史文章 i 最相似的 TopK 篇文章
        # 使用 get 避免个别 item 缺少相似度记录时 KeyError
        sim_items = sorted(
            i2i_sim.get(i, {}).items(),
            key=lambda x: x[1],
            reverse=True
        )[:sim_item_topk]

        for j, wij in sim_items:

            # 排除用户已经点击过的文章
            if j in user_hist_items_:
                continue

            # 3. 文章创建时间差权重（保留原文公式）
            created_time_weight = np.exp(
                0.8 ** np.abs(
                    item_created_time_dict[i]
                    - item_created_time_dict[j]
                )
            )

            # 4. 历史点击位置权重（保留原文公式）
            loc_weight = (
                0.9 ** (len(user_hist_items) - loc)
            )

            # 5. 内容 Embedding 相似度权重
            content_weight = 1.0

            if emb_i2i_sim.get(i, {}).get(j, None) is not None:
                content_weight += emb_i2i_sim[i][j]

            if emb_i2i_sim.get(j, {}).get(i, None) is not None:
                content_weight += emb_i2i_sim[j][i]

            # 6. 累加候选文章得分
            item_rank.setdefault(j, 0)

            item_rank[j] += (
                created_time_weight
                * loc_weight
                * content_weight
                * wij
            )

    # 7. 候选不足时，使用热门文章补全
    if len(item_rank) < recall_item_num:

        for loc, item in enumerate(item_topk_click):

            # 修复原文的 item in item_rank.items()
            # 同时避免重复推荐已点击文章
            if item in user_hist_items_ or item in item_rank:
                continue

            item_rank[item] = -loc - 100

            if len(item_rank) == recall_item_num:
                break

    # 8. 按得分降序返回最终候选
    item_rank = sorted(
        item_rank.items(),
        key=lambda x: x[1],
        reverse=True
    )[:recall_item_num]

    return item_rank

In [27]:
import math
from collections import defaultdict

# ============================================================
# 基于排序训练历史生成 ItemCF 召回
# ============================================================

# 1. 排序训练历史：使用 rank_hist，
rank_click_df = click_trn_hist.copy()

# 与 Task3 一致，对点击时间戳进行归一化
rank_click_df['click_timestamp'] = (
    rank_click_df['click_timestamp']
    - rank_click_df['click_timestamp'].min()
) / (
    rank_click_df['click_timestamp'].max()
    - rank_click_df['click_timestamp'].min()
)


# 2. 准备文章创建时间字典
# 沿用 Task3 中创建时间归一化的处理方式
created_time = article_info_df['created_at_ts']

created_time_norm = (
    created_time - created_time.min()
) / (
    created_time.max() - created_time.min()
)

item_created_time_dict = dict(zip(
    article_info_df['article_id'],
    created_time_norm
))


# 3. 重新计算排序训练阶段的 ItemCF 相似度矩阵

rank_i2i_sim = itemcf_sim(
    rank_click_df,
    item_created_time_dict
)


# 4. 读取文章内容 Embedding 相似度矩阵
# 这份相似度来自文章内容，不依赖留出的用户点击标签
with open(recall_path + 'emb_i2i_sim.pkl', 'rb') as f:
    emb_i2i_sim = pickle.load(f)


# 5. 获取排序训练用户的可见历史
rank_user_item_time_dict = get_user_item_time(
    rank_click_df
)

# 热门文章同样只能根据 rank_hist 统计
rank_item_topk_click = (
    rank_click_df['click_article_id']
    .value_counts()
    .index[:200]
)


# 6. 只对拥有排序训练标签的用户生成候选
trn_users = click_trn_last['user_id'].unique()

# 保持 ItemCF 的召回参数
sim_item_topk = 20
recall_item_num = 50

trn_recall_list_dict = {}

for user in tqdm(trn_users, desc='Ranking train ItemCF'):
    trn_recall_list_dict[user] = item_based_recommend(
        user_id=user,
        user_item_time_dict=rank_user_item_time_dict,
        i2i_sim=rank_i2i_sim,
        sim_item_topk=sim_item_topk,
        recall_item_num=recall_item_num,
        item_topk_click=rank_item_topk_click,
        item_created_time_dict=item_created_time_dict,
        emb_i2i_sim=emb_i2i_sim
    )


# 7. 保存排序训练召回结果
# 文件与 Task3 的验证召回结果分开保存
with open(save_path + 'trn_itemcf_recall_dict.pkl', 'wb') as f:
    pickle.dump(trn_recall_list_dict, f)


# 8. 转换为 DataFrame
trn_recall_list_df = recall_dict_2_df(
    trn_recall_list_dict
)


# 9. 检查结果
print('训练标签用户数:', len(trn_users))
print('生成召回的用户数:', len(trn_recall_list_dict))
print('训练候选数据:', trn_recall_list_df.shape)

assert len(trn_recall_list_dict) == len(trn_users)

assert trn_recall_list_df['user_id'].isin(
    click_trn_last['user_id']
).all()

assert not trn_recall_list_df.duplicated(
    ['user_id', 'sim_item']
).any()

display(trn_recall_list_df.head())


100%|██████████| 179657/179657 [00:09<00:00, 18092.71it/s]


训练标签用户数: 179657
生成召回的用户数: 179657
训练候选数据: (8982850, 3)


,user_id,sim_item,score
0,1,30408,0.962132
1,1,218733,0.367269
2,1,106598,0.334740
3,1,235808,0.325712
4,1,327122,0.324760


In [33]:

# ============================================================
# 给训练、验证数据打标签，并对训练负样本采样
# ============================================================

np.random.seed(42)  # 固定负采样随机种子，保证实验可重复

trn_user_item_label_df, val_user_item_label_df, tst_user_item_label_df = (
    get_user_recall_item_label_df(
        click_trn_hist,
        click_val_hist,
        click_tst_hist,
        click_trn_last,
        click_val_last,
        trn_recall_list_df,
        val_recall_list_df,
        tst_recall_list_df=None
    )
)

print('训练样本:', trn_user_item_label_df.shape)
print('验证样本:', val_user_item_label_df.shape)
print('测试样本:', tst_user_item_label_df)

print('\n训练标签分布:')
print(trn_user_item_label_df['label'].value_counts())

print('\n验证标签分布:')
print(val_user_item_label_df['label'].value_counts())


pos_data_num: 99775 neg_data_num: 8883075 pos/neg: 0.011232033952206866
训练样本: (301170, 4)
验证样本: (2500000, 4)
测试样本: None

训练标签分布:
label
0.0    201395
1.0     99775
Name: count, dtype: int64

验证标签分布:
label
0.0    2469252
1.0      30748
Name: count, dtype: int64


### 将召回数据转换成字典

In [34]:

# ============================================================
# 将召回数据转换成字典
# ============================================================

# 将最终召回的df数据转换成字典的形式做排序特征
def make_tuple_func(group_df):
    row_data = []
    for name, row_df in group_df.iterrows():
        row_data.append((
            int(row_df['sim_item']),
            row_df['score'],
            row_df['label']
        ))

    return row_data


# ============================================================
# 训练数据
# ============================================================

trn_user_item_label_tuples = (
    trn_user_item_label_df
    .groupby('user_id')[['sim_item', 'score', 'label']]
    .apply(make_tuple_func)
    .reset_index()
)

trn_user_item_label_tuples_dict = dict(zip(
    trn_user_item_label_tuples['user_id'],
    trn_user_item_label_tuples[0]
))


# ============================================================
# 验证数据
# ============================================================

if val_user_item_label_df is not None:
    val_user_item_label_tuples = (
        val_user_item_label_df
        .groupby('user_id')[['sim_item', 'score', 'label']]
        .apply(make_tuple_func)
        .reset_index()
    )

    val_user_item_label_tuples_dict = dict(zip(
        val_user_item_label_tuples['user_id'],
        val_user_item_label_tuples[0]
    ))
else:
    val_user_item_label_tuples_dict = None


# ============================================================
# 测试数据
# ============================================================

if tst_user_item_label_df is not None:
    tst_user_item_label_tuples = (
        tst_user_item_label_df
        .groupby('user_id')[['sim_item', 'score', 'label']]
        .apply(make_tuple_func)
        .reset_index()
    )

    tst_user_item_label_tuples_dict = dict(zip(
        tst_user_item_label_tuples['user_id'],
        tst_user_item_label_tuples[0]
    ))
else:
    tst_user_item_label_tuples_dict = None


# ============================================================
# 检查结果
# ============================================================

print('训练用户数:', len(trn_user_item_label_tuples_dict))

if val_user_item_label_tuples_dict is not None:
    print('验证用户数:', len(val_user_item_label_tuples_dict))

print('测试数据:', 
      None if tst_user_item_label_tuples_dict is None
      else len(tst_user_item_label_tuples_dict))

# 检查转换过程中是否丢失候选
assert sum(
    len(items) for items in trn_user_item_label_tuples_dict.values()
) == len(trn_user_item_label_df)

if val_user_item_label_tuples_dict is not None:
    assert sum(
        len(items) for items in val_user_item_label_tuples_dict.values()
    ) == len(val_user_item_label_df)

# 查看一个训练用户的候选列表
sample_user = next(iter(trn_user_item_label_tuples_dict))
print('\n示例用户:', sample_user)
print('候选列表:', trn_user_item_label_tuples_dict[sample_user][:5])


训练用户数: 179657
验证用户数: 50000
测试数据: None

示例用户: 1
候选列表: [(31836, np.float64(-118.0), np.float64(0.0)), (133734, np.float64(0.2447053001170783), np.float64(0.0))]


## 特征工程

### 制作与用户历史行为相关特征

对于每个用户召回的每个商品， 做特征。 具体步骤如下：

- 对于每个用户， 获取最后点击的N个商品的item_id，
  - 对于该用户的每个召回商品， 计算与上面最后N次点击商品的相似度的和(最大， 最小，均值)， 时间差特征，相似性特征，字数差特征，与该用户的相似性特征

In [35]:

# ============================================================
# 制作与用户历史行为相关的特征
# ============================================================

# 下面基于data做历史相关的特征
def create_feature(users_id, recall_list, click_hist_df,
                   articles_info, articles_emb, user_emb=None, N=1):
    """
    基于用户的历史行为做相关特征
    :param users_id: 用户id
    :param recall_list: 对于每个用户召回的候选文章列表
    :param click_hist_df: 用户的历史点击信息
    :param articles_info: 文章信息
    :param articles_emb: 文章的embedding向量，可以使用内容、W2V等向量
    :param user_emb: 用户的embedding向量，没有可以不使用
    :param N: 最近的N次点击，默认1
    """

    # 建立一个二维列表保存结果，后面要转成DataFrame
    all_user_feas = []
    i = 0

    for user_id in tqdm(users_id):

        # 该用户的最后N次点击
        hist_user_items = click_hist_df[
            click_hist_df['user_id'] == user_id
        ]['click_article_id'][-N:]

        # 遍历该用户的召回列表
        # 修改：按score降序，避免负采样后的拼接顺序影响rank
        sorted_recall = sorted(
            recall_list[user_id],
            key=lambda x: x[1],
            reverse=True
        )

        for rank, (article_id, score, label) in enumerate(sorted_recall):

            # 该文章建立时间、字数
            a_create_time = articles_info[
                articles_info['article_id'] == article_id
            ]['created_at_ts'].values[0]

            a_words_count = articles_info[
                articles_info['article_id'] == article_id
            ]['words_count'].values[0]

            single_user_fea = [user_id, article_id]

            # 计算与最后点击文章的相似性、创建时间差、字数差
            sim_fea = []
            time_fea = []
            word_fea = []

            # 遍历用户的最后N次点击文章
            for hist_item in hist_user_items:

                b_create_time = articles_info[
                    articles_info['article_id'] == hist_item
                ]['created_at_ts'].values[0]

                b_words_count = articles_info[
                    articles_info['article_id'] == hist_item
                ]['words_count'].values[0]

                sim_fea.append(
                    np.dot(
                        articles_emb[hist_item],
                        articles_emb[article_id]
                    )
                )

                time_fea.append(
                    abs(a_create_time - b_create_time)
                )

                word_fea.append(
                    abs(a_words_count - b_words_count)
                )

            single_user_fea.extend(sim_fea)    # 相似性特征
            single_user_fea.extend(time_fea)   # 时间差特征
            single_user_fea.extend(word_fea)   # 字数差特征

            # 相似性的统计特征
            single_user_fea.extend([
                max(sim_fea),
                min(sim_fea),
                sum(sim_fea),
                sum(sim_fea) / len(sim_fea)
            ])

            # 如果有用户向量，计算用户与候选文章的相似性
            if user_emb:
                single_user_fea.append(
                    np.dot(
                        user_emb[user_id],
                        articles_emb[article_id]
                    )
                )

            single_user_fea.extend([score, rank, label])

            # 加入总表
            all_user_feas.append(single_user_fea)

    # 定义列名
    id_cols = ['user_id', 'click_article_id']
    sim_cols = ['sim' + str(i) for i in range(N)]
    time_cols = ['time_diff' + str(i) for i in range(N)]
    word_cols = ['word_diff' + str(i) for i in range(N)]
    sat_cols = ['sim_max', 'sim_min', 'sim_sum', 'sim_mean']
    user_item_sim_cols = ['user_item_sim'] if user_emb else []
    user_score_rank_label = ['score', 'rank', 'label']

    cols = (
        id_cols
        + sim_cols
        + time_cols
        + word_cols
        + sat_cols
        + user_item_sim_cols
        + user_score_rank_label
    )

    # 转成DataFrame
    df = pd.DataFrame(all_user_feas, columns=cols)

    return df


In [37]:

# 读取 Task3 已保存的文章内容 Embedding
with open(recall_path + 'item_content_emb.pkl', 'rb') as f:
    item_content_emb_dict = pickle.load(f)

print('文章信息表:', article_info_df.shape)
print('内容 Embedding 数量:', len(item_content_emb_dict))

# 检查文章信息及 Embedding 是否能够覆盖当前候选和历史
candidate_items = (
    set(trn_user_item_label_df['sim_item'])
    | set(val_user_item_label_df['sim_item'])
)

history_items = (
    set(click_trn_hist['click_article_id'])
    | set(click_val_hist['click_article_id'])
)

needed_items = candidate_items | history_items

missing_info = needed_items - set(article_info_df['article_id'])
missing_emb = needed_items - set(item_content_emb_dict)

print('缺少文章属性的文章数:', len(missing_info))
print('缺少内容 Embedding 的文章数:', len(missing_emb))

assert not missing_info, "部分文章缺少属性信息"
assert not missing_emb, "部分文章缺少内容 Embedding"


文章信息表: (364047, 4)
内容 Embedding 数量: 364047
缺少文章属性的文章数: 0
缺少内容 Embedding 的文章数: 0


In [38]:

# 先用少量训练用户验证 create_feature() 的输出
sample_users = list(
    trn_user_item_label_tuples_dict.keys()
)[:3]

sample_feats_df = create_feature(
    sample_users,
    trn_user_item_label_tuples_dict,
    click_trn_hist,
    article_info_df,
    item_content_emb_dict,
    N=1
)

print('样本特征维度:', sample_feats_df.shape)
print('生成的特征列:')
print(sample_feats_df.columns.tolist())

display(sample_feats_df.head())

assert sample_feats_df.shape[1] == 12
assert sample_feats_df['label'].isin([0, 1]).all()
assert sample_feats_df[
    ['sim0', 'time_diff0', 'word_diff0', 'score', 'rank']
].notna().all().all()


100%|██████████| 3/3 [00:00<00:00, 212.56it/s]

样本特征维度: (5, 12)
生成的特征列:
['user_id', 'click_article_id', 'sim0', 'time_diff0', 'word_diff0', 'sim_max', 'sim_min', 'sim_sum', 'sim_mean', 'score', 'rank', 'label']


,user_id,click_article_id,sim0,time_diff0,word_diff0,sim_max,sim_min,sim_sum,sim_mean,score,rank,label
0,1,133734,0.113535,26620654000,21,0.113535,0.113535,0.113535,0.113535,0.244705,0,0.0
1,1,31836,0.150834,626430000,1,0.150834,0.150834,0.150834,0.150834,-118.000000,1,0.0
2,2,233717,0.238539,1087514000,21,0.238539,0.238539,0.238539,0.238539,-108.000000,0,0.0
3,3,36162,0.646486,5148000,17,0.646486,0.646486,0.646486,0.646486,0.894016,0,1.0
4,3,540,0.316301,86330217000,17,0.316301,0.316301,0.316301,0.316301,0.276184,1,0.0


In [39]:

# ============================================================
# 获取训练、验证及测试数据中召回文章的相关特征
# ============================================================

# 排序训练：使用 rank_hist，对应预测目标 rank_targets
trn_user_item_feats_df = create_feature(
    trn_user_item_label_tuples_dict.keys(),
    trn_user_item_label_tuples_dict,
    click_trn_hist,
    article_info_df,
    item_content_emb_dict
)

# 最终验证：使用 query_clicks，对应预测目标 targets
if val_user_item_label_tuples_dict is not None:
    val_user_item_feats_df = create_feature(
        val_user_item_label_tuples_dict.keys(),
        val_user_item_label_tuples_dict,
        click_val_hist,
        article_info_df,
        item_content_emb_dict
    )
else:
    val_user_item_feats_df = None

# 当前为 offline=True，不生成 TestA 特征
tst_user_item_feats_df = None


# ============================================================
# 保存特征结果
# ============================================================

os.makedirs(save_path, exist_ok=True)

trn_user_item_feats_df.to_csv(
    save_path + 'trn_user_item_feats_df.csv',
    index=False
)

if val_user_item_feats_df is not None:
    val_user_item_feats_df.to_csv(
        save_path + 'val_user_item_feats_df.csv',
        index=False
    )

print('训练特征:', trn_user_item_feats_df.shape)

if val_user_item_feats_df is not None:
    print('验证特征:', val_user_item_feats_df.shape)


100%|██████████| 50000/50000 [44:25<00:00, 18.76it/s]  


训练特征: (301170, 12)
验证特征: (2500000, 12)


## 用户和文章特征

### 用户相关特征

这一块，正式进行特征工程，既要拼接上已有的特征， 也会做更多的特征出来，我们来梳理一下已有的特征和可构造特征：

1.文章自身的特征， 文章字数，文章创建时间， 文章的embedding （articles表中)

2.用户点击环境特征， 那些设备的特征(这个在df中)

3.对于用户和商品还可以构造的特征：

- 基于用户的点击文章次数和点击时间构造可以表现用户活跃度的特征

- 基于文章被点击次数和时间构造可以反映文章热度的特征

- 用户的时间统计特征： 根据其点击的历史文章列表的点击时间和文章的创建时间做统计特征，比如求均值， 这个可以反映用户对于文章时效的偏好

- 用户的主题爱好特征， 对于用户点击的历史文章主题进行一个统计， 然后对于当前文章看看是否属于用户已经点击过的主题

- 用户的字数爱好特征， 对于用户点击的历史文章的字数统计， 求一个均值

In [40]:

# ============================================================
# 用户和文章特征：数据准备
# ============================================================

# 复用已经读取的文章信息
articles = article_info_df

# 原始点击日志包含设备、系统、地区等9个字段
# 公共数据划分只保留3列，这里按照点击事件恢复其他字段
raw_clicks = train_clicks.drop_duplicates(
    subset=click_cols
)

# 排序训练：只允许使用 rank_hist（A）
all_data_trn = click_trn_hist.merge(
    raw_clicks,
    on=click_cols,
    how='left',
    validate='one_to_one'
)

# 最终验证：使用 fit_clicks（AB）
# 包含验证用户的全部可见历史，以及其他拟合用户的点击
all_data_val = click_trn.merge(
    raw_clicks,
    on=click_cols,
    how='left',
    validate='one_to_one'
)

# 沿用原文的内存优化
all_data_trn = reduce_mem(all_data_trn)
all_data_val = reduce_mem(all_data_val)

# 拼接文章属性
all_data_trn = all_data_trn.merge(
    articles,
    left_on='click_article_id',
    right_on='article_id',
    how='left',
    validate='many_to_one'
)

all_data_val = all_data_val.merge(
    articles,
    left_on='click_article_id',
    right_on='article_id',
    how='left',
    validate='many_to_one'
)

# 检查合并是否保留全部点击事件
assert len(all_data_trn) == len(click_trn_hist)
assert len(all_data_val) == len(click_trn)

assert all_data_trn['article_id'].notna().all()
assert all_data_val['article_id'].notna().all()

assert all_data_trn['click_environment'].notna().all()
assert all_data_val['click_environment'].notna().all()

print('训练特征原始数据:', all_data_trn.shape)
print('验证特征原始数据:', all_data_val.shape)

display(all_data_trn.head())


-- Mem. usage decreased to 18.53 Mb (69.4% reduction),time spend:0.00 min
-- Mem. usage decreased to 22.29 Mb (69.4% reduction),time spend:0.00 min
训练特征原始数据: (882966, 13)
验证特征原始数据: (1062623, 13)


,user_id,click_article_id,click_timestamp,click_environment,click_deviceGroup,click_os,click_country,click_region,click_referrer_type,article_id,category_id,created_at_ts,words_count
0,0,30760,1508211672520,4,1,17,1,25,2,30760,26,1508185091000,162
1,1,289197,1508211316889,4,1,17,1,25,6,289197,418,1508179909000,176
2,2,36162,1508211438695,4,3,20,1,25,2,36162,43,1508177171000,205
3,3,50644,1508211359672,4,3,2,1,25,2,50644,99,1508182319000,188
4,4,42567,1508211625466,4,1,12,1,16,1,42567,67,1508147384000,265


### 点击时间和点击文章次数，区分用户活跃度

如果某个用户点击文章之间的时间间隔比较小， 同时点击的文章次数很多的话， 那么我们认为这种用户一般就是活跃用户, 当然衡量用户活跃度的方式可能多种多样， 这里我们只提供其中一种，我们写一个函数， 得到可以衡量用户活跃度的特征，逻辑如下：

1. 首先根据用户user_id分组， 对于每个用户，计算点击文章的次数， 两两点击文章时间间隔的均值

2. 把点击次数取倒数和时间间隔的均值统一归一化，然后两者相加合并，该值越小， 说明用户越活跃

3. 注意， 上面两两点击文章的时间间隔均值， 会出现如果用户只点击了一次的情况，这时候时间间隔均值那里会出现空值， 对于这种情况最后特征那里给个大数进行区分

这个的衡量标准就是先把点击的次数取到数然后归一化， 然后点击的时间差归一化， 然后两者相加进行合并， 该值越小， 说明被点击的次数越多， 且间隔时间短。

In [41]:

# ============================================================
# 分析点击时间和点击次数，区分用户活跃度
# ============================================================

def active_level(all_data, cols,
                 norm_stats=None, return_norm_stats=False):
    """
    制作区分用户活跃度的特征
    :param all_data: 数据集
    :param cols: 用到的特征列
    """

    data = all_data[cols].copy()
    data.sort_values(
        ['user_id', 'click_timestamp'],
        inplace=True
    )

    # 按用户分组，统计点击次数和点击时间序列
    user_act = data.groupby(
        'user_id', as_index=False
    ).agg(
        click_size=('click_article_id', 'size'),
        click_timestamp=('click_timestamp', list)
    )

    # 计算时间间隔的均值
    def time_diff_mean(l):
        if len(l) == 1:
            return 1
        else:
            return np.mean([
                j - i for i, j in zip(l[:-1], l[1:])
            ])

    user_act['time_diff_mean'] = user_act[
        'click_timestamp'
    ].apply(lambda x: time_diff_mean(x))

    # 点击次数取倒数
    user_act['click_size'] = 1 / user_act['click_size']

    # 记录训练数据的归一化范围
    if norm_stats is None:
        norm_stats = {
            'click_size': (
                user_act['click_size'].min(),
                user_act['click_size'].max()
            ),
            'time_diff_mean': (
                user_act['time_diff_mean'].min(),
                user_act['time_diff_mean'].max()
            )
        }

    # 两者归一化
    for col in ['click_size', 'time_diff_mean']:
        c_min, c_max = norm_stats[col]

        if c_max > c_min:
            user_act[col] = (
                user_act[col] - c_min
            ) / (c_max - c_min)
        else:
            user_act[col] = 0.0

    # 两个特征相加，得到用户活跃度
    user_act['active_level'] = (
        user_act['click_size']
        + user_act['time_diff_mean']
    )

    user_act['user_id'] = user_act['user_id'].astype('int')
    del user_act['click_timestamp']

    if return_norm_stats:
        return user_act, norm_stats

    return user_act


In [42]:

# ============================================================
# 计算训练与验证用户活跃度
# ============================================================

act_cols = [
    'user_id',
    'click_article_id',
    'click_timestamp'
]

# 训练集：生成活跃度特征并记录归一化范围
user_act_fea, act_norm_stats = active_level(
    all_data_trn,
    act_cols,
    return_norm_stats=True
)

# 验证集：使用训练阶段的归一化范围
val_user_act_fea = active_level(
    all_data_val,
    act_cols,
    norm_stats=act_norm_stats
)

# 检查结果
print('训练用户活跃度特征:', user_act_fea.shape)
print('验证用户活跃度特征:', val_user_act_fea.shape)

assert user_act_fea['user_id'].nunique() == (
    click_trn_hist['user_id'].nunique()
)

assert val_user_act_fea['user_id'].nunique() == (
    click_trn['user_id'].nunique()
)

assert np.isfinite(
    user_act_fea['active_level']
).all()

assert np.isfinite(
    val_user_act_fea['active_level']
).all()

display(user_act_fea.head())
display(val_user_act_fea.head())


训练用户活跃度特征: (200000, 4)
验证用户活跃度特征: (200000, 4)


,user_id,click_size,time_diff_mean,active_level
0,0,1.0,0.0,1.0
1,1,1.0,0.0,1.0
2,2,1.0,0.0,1.0
3,3,1.0,0.0,1.0
4,4,1.0,0.0,1.0


,user_id,click_size,time_diff_mean,active_level
0,0,1.000000,0.000000,1.000000
1,1,0.497908,0.000027,0.497935
2,2,0.497908,0.000027,0.497935
3,3,0.497908,0.000027,0.497935
4,4,1.000000,0.000000,1.000000


### 点击时间和被点击文章次数，文章热度特征

和上面同样的思路， 如果一篇文章在很短的时间间隔之内被点击了很多次， 说明文章比较热门，实现的逻辑和上面的基本一致， 只不过这里是按照点击的文章进行分组：

1. 根据文章进行分组， 对于每篇文章的用户， 计算点击的时间间隔

2. 将用户的数量取倒数， 然后用户的数量和时间间隔归一化， 然后相加得到热度特征， 该值越小， 说明被点击的次数越大且时间间隔越短， 文章比较热

这只是一种判断文章热度的方法。

In [43]:

# ============================================================
# 分析点击时间和被点击文章的次数，衡量文章热度特征
# ============================================================

def hot_level(all_data, cols,
              norm_stats=None, return_norm_stats=False):
    """
    制作衡量文章热度的特征
    :param all_data: 数据集
    :param cols: 用到的特征列
    """

    data = all_data[cols].copy()
    data.sort_values(
        ['click_article_id', 'click_timestamp'],
        inplace=True
    )

    # 根据文章进行分组，统计点击次数和点击时间序列
    article_hot = data.groupby(
        'click_article_id', as_index=False
    ).agg(
        user_num=('user_id', 'size'),
        click_timestamp=('click_timestamp', list)
    )

    # 计算被点击时间间隔的均值
    def time_diff_mean(l):
        if len(l) == 1:
            return 1
        else:
            return np.mean([
                j - i for i, j in list(zip(l[:-1], l[1:]))
            ])

    article_hot['time_diff_mean'] = article_hot[
        'click_timestamp'
    ].apply(lambda x: time_diff_mean(x))

    # 点击次数取倒数
    article_hot['user_num'] = 1 / article_hot['user_num']

    # 保存训练阶段的归一化范围
    if norm_stats is None:
        norm_stats = {
            'user_num': (
                article_hot['user_num'].min(),
                article_hot['user_num'].max()
            ),
            'time_diff_mean': (
                article_hot['time_diff_mean'].min(),
                article_hot['time_diff_mean'].max()
            )
        }

    # 两者归一化
    for col in ['user_num', 'time_diff_mean']:
        c_min, c_max = norm_stats[col]

        if c_max > c_min:
            article_hot[col] = (
                article_hot[col] - c_min
            ) / (c_max - c_min)
        else:
            article_hot[col] = 0.0

    # 两个特征相加，得到文章热度
    article_hot['hot_level'] = (
        article_hot['user_num']
        + article_hot['time_diff_mean']
    )

    article_hot['click_article_id'] = (
        article_hot['click_article_id'].astype('int')
    )

    del article_hot['click_timestamp']

    if return_norm_stats:
        return article_hot, norm_stats

    return article_hot


In [44]:

# ============================================================
# 生成训练、验证文章热度特征
# ============================================================

hot_cols = [
    'user_id',
    'click_article_id',
    'click_timestamp'
]

# 排序训练：基于 rank_hist 计算热度
article_hot_fea, hot_norm_stats = hot_level(
    all_data_trn,
    hot_cols,
    return_norm_stats=True
)

# 最终验证：基于 fit_clicks 计算热度
# 使用训练阶段的归一化范围
val_article_hot_fea = hot_level(
    all_data_val,
    hot_cols,
    norm_stats=hot_norm_stats
)

# ============================================================
# 检查结果
# ============================================================

print('训练文章热度:', article_hot_fea.shape)
print('验证文章热度:', val_article_hot_fea.shape)

assert article_hot_fea['click_article_id'].is_unique
assert val_article_hot_fea['click_article_id'].is_unique

assert article_hot_fea['hot_level'].notna().all()
assert val_article_hot_fea['hot_level'].notna().all()

display(article_hot_fea.head())
display(val_article_hot_fea.head())


训练文章热度: (25824, 4)
验证文章热度: (29986, 4)


,click_article_id,user_num,time_diff_mean,hot_level
0,69,1.0,0.0,1.0
1,84,1.0,0.0,1.0
2,94,1.0,0.0,1.0
3,142,1.0,0.0,1.0
4,163,1.0,0.0,1.0


,click_article_id,user_num,time_diff_mean,hot_level
0,3,1.0,0.0,1.0
1,69,1.0,0.0,1.0
2,84,1.0,0.0,1.0
3,94,1.0,0.0,1.0
4,125,1.0,0.0,1.0


### 用户的系列习惯

基于原来的日志表做一个类似于article的那种DataFrame， 存放用户特有的信息, 主要包括点击习惯， 爱好特征之类的

- 用户的设备习惯， 这里取最常用的设备（众数）

- 用户的时间习惯： 根据其点击过得历史文章的时间来做一个统计（这个感觉最好是把时间戳里的时间特征的h特征提出来，看看用户习惯一天的啥时候点击文章）， 但这里先用转换的时间吧， 求个均值

- 用户的爱好特征， 对于用户点击的历史文章主题进行用户的爱好判别， 更偏向于哪几个主题， 这个最好是multi-hot进行编码， 先试试行不

- 用户文章的字数差特征， 用户的爱好文章的字数习惯

这些就是对用户进行分组， 然后统计即可

In [45]:

# ============================================================
# 用户的设备习惯
# ============================================================

def device_fea(all_data, cols):
    """
    制作用户的设备特征
    :param all_data: 数据集
    :param cols: 用到的特征列
    """
    user_device_info = all_data[cols]

    # 用众数来表示每个用户的设备信息
    user_device_info = user_device_info.groupby('user_id').agg(
        lambda x: x.value_counts().index[0]
    ).reset_index()

    return user_device_info


# ============================================================
# 生成训练、验证用户设备特征
# ============================================================

# 设备特征(这里时间会比较长)
device_cols = [
    'user_id',
    'click_environment',
    'click_deviceGroup',
    'click_os',
    'click_country',
    'click_region',
    'click_referrer_type'
]

# 排序训练：基于 rank_hist
user_device_info = device_fea(
    all_data_trn,
    device_cols
)

# 最终验证：基于 fit_clicks
val_user_device_info = device_fea(
    all_data_val,
    device_cols
)


# ============================================================
# 检查结果
# ============================================================

print('训练设备特征:', user_device_info.shape)
print('验证设备特征:', val_user_device_info.shape)

assert user_device_info['user_id'].is_unique
assert val_user_device_info['user_id'].is_unique

assert user_device_info[device_cols].notna().all().all()
assert val_user_device_info[device_cols].notna().all().all()

display(user_device_info.head())
display(val_user_device_info.head())


训练设备特征: (200000, 7)
验证设备特征: (200000, 7)


,user_id,click_environment,click_deviceGroup,click_os,click_country,click_region,click_referrer_type
0,0,4,1,17,1,25,2
1,1,4,1,17,1,25,6
2,2,4,3,20,1,25,2
3,3,4,3,2,1,25,2
4,4,4,1,12,1,16,1


,user_id,click_environment,click_deviceGroup,click_os,click_country,click_region,click_referrer_type
0,0,4,1,17,1,25,2
1,1,4,1,17,1,25,6
2,2,4,3,20,1,25,2
3,3,4,3,2,1,25,2
4,4,4,1,12,1,16,1


In [46]:

# ============================================================
# 用户的时间习惯
# ============================================================

def user_time_hob_fea(all_data, cols, mm=None, return_mm=False):
    """
    制作用户的时间习惯特征
    :param all_data: 数据集
    :param cols: 用到的特征列
    """

    user_time_hob_info = all_data[cols].copy()

    # 先把时间戳进行归一化
    time_cols = ['click_timestamp', 'created_at_ts']

    if mm is None:
        mm = MinMaxScaler()
        user_time_hob_info[time_cols] = mm.fit_transform(
            user_time_hob_info[time_cols]
        )
    else:
        user_time_hob_info[time_cols] = mm.transform(
            user_time_hob_info[time_cols]
        )

    # 按用户计算归一化时间的均值
    user_time_hob_info = user_time_hob_info.groupby(
        'user_id'
    ).agg('mean').reset_index()

    user_time_hob_info.rename(
        columns={
            'click_timestamp': 'user_time_hob1',
            'created_at_ts': 'user_time_hob2'
        },
        inplace=True
    )

    if return_mm:
        return user_time_hob_info, mm

    return user_time_hob_info


# ============================================================
# 生成训练、验证用户时间习惯特征
# ============================================================

user_time_hob_cols = [
    'user_id',
    'click_timestamp',
    'created_at_ts'
]

# 排序训练：基于 rank_hist，拟合归一化参数
user_time_hob_info, time_mm = user_time_hob_fea(
    all_data_trn,
    user_time_hob_cols,
    return_mm=True
)

# 最终验证：基于 fit_clicks，沿用训练阶段的归一化参数
val_user_time_hob_info = user_time_hob_fea(
    all_data_val,
    user_time_hob_cols,
    mm=time_mm
)


# ============================================================
# 检查结果
# ============================================================

print('训练时间习惯特征:', user_time_hob_info.shape)
print('验证时间习惯特征:', val_user_time_hob_info.shape)

assert user_time_hob_info['user_id'].is_unique
assert val_user_time_hob_info['user_id'].is_unique

assert user_time_hob_info.notna().all().all()
assert val_user_time_hob_info.notna().all().all()

display(user_time_hob_info.head())
display(val_user_time_hob_info.head())


训练时间习惯特征: (200000, 3)
验证时间习惯特征: (200000, 3)


,user_id,user_time_hob1,user_time_hob2
0,0,0.330771,0.993047
1,1,0.330672,0.993032
2,2,0.330706,0.993024
3,3,0.330684,0.993039
4,4,0.330758,0.992937


,user_id,user_time_hob1,user_time_hob2
0,0,0.330771,0.993047
1,1,0.330676,0.992977
2,2,0.330710,0.992277
3,3,0.330688,0.993031
4,4,0.330758,0.992937


In [47]:

# ============================================================
# 用户的主题爱好
# ============================================================

def user_cat_hob_fea(all_data, cols):
    """
    用户的主题爱好
    :param all_data: 数据集
    :param cols: 用到的特征列
    """
    user_category_hob_info = all_data[cols]

    # 按用户分组，将点击文章的类别转成列表
    user_category_hob_info = user_category_hob_info.groupby(
        'user_id'
    )['category_id'].agg(list).reset_index()

    user_cat_hob_info = pd.DataFrame()
    user_cat_hob_info['user_id'] = user_category_hob_info['user_id']
    user_cat_hob_info['cate_list'] = user_category_hob_info['category_id']

    return user_cat_hob_info


# ============================================================
# 生成训练、验证用户主题偏好
# ============================================================

user_category_hob_cols = ['user_id', 'category_id']

# 排序训练：基于 rank_hist
user_cat_hob_info = user_cat_hob_fea(
    all_data_trn,
    user_category_hob_cols
)

# 最终验证：基于 fit_clicks
val_user_cat_hob_info = user_cat_hob_fea(
    all_data_val,
    user_category_hob_cols
)


# ============================================================
# 检查结果
# ============================================================

print('训练用户主题特征:', user_cat_hob_info.shape)
print('验证用户主题特征:', val_user_cat_hob_info.shape)

assert user_cat_hob_info['user_id'].is_unique
assert val_user_cat_hob_info['user_id'].is_unique

assert user_cat_hob_info['cate_list'].map(
    lambda x: isinstance(x, list) and len(x) > 0
).all()

assert val_user_cat_hob_info['cate_list'].map(
    lambda x: isinstance(x, list) and len(x) > 0
).all()

display(user_cat_hob_info.head())
display(val_user_cat_hob_info.head())


训练用户主题特征: (200000, 2)
验证用户主题特征: (200000, 2)


,user_id,cate_list
0,0,[26]
1,1,[418]
2,2,[43]
3,3,[99]
4,4,[67]


,user_id,cate_list
0,0,[26]
1,1,"[418, 133]"
2,2,"[43, 297]"
3,3,"[99, 43]"
4,4,[67]


In [48]:

# ============================================================
# 用户的字数偏好特征
# ============================================================

# 排序训练：基于 rank_hist
user_wcou_info = all_data_trn.groupby(
    'user_id'
)['words_count'].agg('mean').reset_index()

user_wcou_info.rename(
    columns={'words_count': 'words_hbo'},
    inplace=True
)


# 最终验证：基于 fit_clicks
val_user_wcou_info = all_data_val.groupby(
    'user_id'
)['words_count'].agg('mean').reset_index()

val_user_wcou_info.rename(
    columns={'words_count': 'words_hbo'},
    inplace=True
)


# ============================================================
# 检查结果
# ============================================================

print('训练用户字数偏好:', user_wcou_info.shape)
print('验证用户字数偏好:', val_user_wcou_info.shape)

assert user_wcou_info['user_id'].is_unique
assert val_user_wcou_info['user_id'].is_unique

assert user_wcou_info['words_hbo'].notna().all()
assert val_user_wcou_info['words_hbo'].notna().all()

display(user_wcou_info.head())
display(val_user_wcou_info.head())


训练用户字数偏好: (200000, 2)
验证用户字数偏好: (200000, 2)


,user_id,words_hbo
0,0,162.0
1,1,176.0
2,2,205.0
3,3,188.0
4,4,265.0


,user_id,words_hbo
0,0,162.0
1,1,169.0
2,2,210.0
3,3,196.5
4,4,265.0


### 用户的信息特征合并保存

In [49]:

# ============================================================
# 用户的信息特征合并保存
# ============================================================

# 所有表进行合并
# 排序训练：基于 rank_hist 生成的用户特征
user_info = pd.merge(user_act_fea, user_device_info, on='user_id')
user_info = user_info.merge(user_time_hob_info, on='user_id')
user_info = user_info.merge(user_cat_hob_info, on='user_id')
user_info = user_info.merge(user_wcou_info, on='user_id')


# 最终验证：基于 fit_clicks 生成的用户特征
val_user_info = pd.merge(
    val_user_act_fea, val_user_device_info, on='user_id'
)
val_user_info = val_user_info.merge(
    val_user_time_hob_info, on='user_id'
)
val_user_info = val_user_info.merge(
    val_user_cat_hob_info, on='user_id'
)
val_user_info = val_user_info.merge(
    val_user_wcou_info, on='user_id'
)


# ============================================================
# 检查合并结果
# ============================================================

print('训练用户画像:', user_info.shape)
print('验证用户画像:', val_user_info.shape)

assert user_info['user_id'].is_unique
assert val_user_info['user_id'].is_unique

assert len(user_info) == click_trn_hist['user_id'].nunique()
assert len(val_user_info) == click_trn['user_id'].nunique()

assert user_info.columns.tolist() == val_user_info.columns.tolist()

display(user_info.head())
display(val_user_info.head())


# ============================================================
# 保存用户特征，以后可以直接读取
# ============================================================

user_info.to_csv(
    save_path + 'user_info.csv',
    index=False
)

val_user_info.to_csv(
    save_path + 'val_user_info.csv',
    index=False
)

print('用户画像保存完成')


训练用户画像: (200000, 14)
验证用户画像: (200000, 14)


,user_id,click_size,time_diff_mean,active_level,click_environment,click_deviceGroup,click_os,click_country,click_region,click_referrer_type,user_time_hob1,user_time_hob2,cate_list,words_hbo
0,0,1.0,0.0,1.0,4,1,17,1,25,2,0.330771,0.993047,[26],162.0
1,1,1.0,0.0,1.0,4,1,17,1,25,6,0.330672,0.993032,[418],176.0
2,2,1.0,0.0,1.0,4,3,20,1,25,2,0.330706,0.993024,[43],205.0
3,3,1.0,0.0,1.0,4,3,2,1,25,2,0.330684,0.993039,[99],188.0
4,4,1.0,0.0,1.0,4,1,12,1,16,1,0.330758,0.992937,[67],265.0


,user_id,click_size,time_diff_mean,active_level,click_environment,click_deviceGroup,click_os,click_country,click_region,click_referrer_type,user_time_hob1,user_time_hob2,cate_list,words_hbo
0,0,1.000000,0.000000,1.000000,4,1,17,1,25,2,0.330771,0.993047,[26],162.0
1,1,0.497908,0.000027,0.497935,4,1,17,1,25,6,0.330676,0.992977,"[418, 133]",169.0
2,2,0.497908,0.000027,0.497935,4,3,20,1,25,2,0.330710,0.992277,"[43, 297]",210.0
3,3,0.497908,0.000027,0.497935,4,3,2,1,25,2,0.330688,0.993031,"[99, 43]",196.5
4,4,1.000000,0.000000,1.000000,4,1,12,1,16,1,0.330758,0.992937,[67],265.0


用户画像保存完成


## 用户特征直接读入

按照user_id合并用户特征

In [50]:

# ============================================================
# 用户特征直接读入
# ============================================================

import ast

# 读取训练和验证阶段的用户画像
user_info = pd.read_csv(
    save_path + 'user_info.csv'
)

val_user_info = pd.read_csv(
    save_path + 'val_user_info.csv'
)

# CSV读取后cate_list变为字符串，需要恢复成Python列表
user_info['cate_list'] = user_info['cate_list'].apply(
    ast.literal_eval
)

val_user_info['cate_list'] = val_user_info['cate_list'].apply(
    ast.literal_eval
)


# ============================================================
# 读取此前生成的候选文章相关特征
# ============================================================

if os.path.exists(save_path + 'trn_user_item_feats_df.csv'):
    trn_user_item_feats_df = pd.read_csv(
        save_path + 'trn_user_item_feats_df.csv',
        dtype={'user_id': 'int64', 'click_article_id': 'int64'}
    )
else:
    raise FileNotFoundError('训练候选特征文件不存在')

if os.path.exists(save_path + 'val_user_item_feats_df.csv'):
    val_user_item_feats_df = pd.read_csv(
        save_path + 'val_user_item_feats_df.csv',
        dtype={'user_id': 'int64', 'click_article_id': 'int64'}
    )
else:
    val_user_item_feats_df = None

# 当前offline=True，不使用TestA特征
tst_user_item_feats_df = None


# ============================================================
# 拼上用户特征
# ============================================================

# 记录合并前的数据行数
trn_rows = len(trn_user_item_feats_df)
val_rows = (
    len(val_user_item_feats_df)
    if val_user_item_feats_df is not None else 0
)

# 排序训练：使用rank_hist对应的用户画像
trn_user_item_feats_df = trn_user_item_feats_df.merge(
    user_info,
    on='user_id',
    how='left',
    validate='many_to_one'
)

# 最终验证：使用fit_clicks对应的用户画像
if val_user_item_feats_df is not None:
    val_user_item_feats_df = val_user_item_feats_df.merge(
        val_user_info,
        on='user_id',
        how='left',
        validate='many_to_one'
    )
else:
    val_user_item_feats_df = None


# ============================================================
# 检查合并结果
# ============================================================

assert len(trn_user_item_feats_df) == trn_rows
assert trn_user_item_feats_df['active_level'].notna().all()

if val_user_item_feats_df is not None:
    assert len(val_user_item_feats_df) == val_rows
    assert val_user_item_feats_df['active_level'].notna().all()

print('训练特征:', trn_user_item_feats_df.shape)

if val_user_item_feats_df is not None:
    print('验证特征:', val_user_item_feats_df.shape)

print('\n训练特征列:')
print(trn_user_item_feats_df.columns.tolist())

display(trn_user_item_feats_df.head())


训练特征: (301170, 25)
验证特征: (2500000, 25)

训练特征列:
['user_id', 'click_article_id', 'sim0', 'time_diff0', 'word_diff0', 'sim_max', 'sim_min', 'sim_sum', 'sim_mean', 'score', 'rank', 'label', 'click_size', 'time_diff_mean', 'active_level', 'click_environment', 'click_deviceGroup', 'click_os', 'click_country', 'click_region', 'click_referrer_type', 'user_time_hob1', 'user_time_hob2', 'cate_list', 'words_hbo']


,user_id,click_article_id,sim0,time_diff0,word_diff0,sim_max,sim_min,sim_sum,sim_mean,score,...,click_environment,click_deviceGroup,click_os,click_country,click_region,click_referrer_type,user_time_hob1,user_time_hob2,cate_list,words_hbo
0,1,133734,0.113535,26620654000,21,0.113535,0.113535,0.113535,0.113535,0.244705,...,4,1,17,1,25,6,0.330672,0.993032,[418],176.0
1,1,31836,0.150834,626430000,1,0.150834,0.150834,0.150834,0.150834,-118.000000,...,4,1,17,1,25,6,0.330672,0.993032,[418],176.0
2,2,233717,0.238539,1087514000,21,0.238539,0.238539,0.238539,0.238539,-108.000000,...,4,3,20,1,25,2,0.330706,0.993024,[43],205.0
3,3,36162,0.646486,5148000,17,0.646486,0.646486,0.646486,0.646486,0.894016,...,4,3,2,1,25,2,0.330684,0.993039,[99],188.0
4,3,540,0.316301,86330217000,17,0.316301,0.316301,0.316301,0.316301,0.276184,...,4,3,2,1,25,2,0.330684,0.993039,[99],188.0


### 文章特征直接读入

按照文章ID合并

In [51]:

# ============================================================
# 文章的特征直接读入
# ============================================================

# 前面已经读取并进行内存优化，无需重复读取
articles = article_info_df

# 检查文章ID是否唯一
assert articles['article_id'].is_unique

# 检查文章信息能否覆盖所有训练、验证候选
candidate_ids = set(
    trn_user_item_feats_df['click_article_id'].unique()
)

if val_user_item_feats_df is not None:
    candidate_ids.update(
        val_user_item_feats_df['click_article_id'].unique()
    )

missing_items = candidate_ids - set(articles['article_id'])

if missing_items:
    raise ValueError(
        f'有 {len(missing_items)} 篇候选文章缺少属性信息，'
        f'示例：{list(missing_items)[:5]}'
    )

# 记录合并前的行数
trn_rows = len(trn_user_item_feats_df)
val_rows = (
    len(val_user_item_feats_df)
    if val_user_item_feats_df is not None else 0
)

# ============================================================
# 拼上文章特征
# ============================================================

trn_user_item_feats_df = trn_user_item_feats_df.merge(
    articles,
    left_on='click_article_id',
    right_on='article_id',
    validate='many_to_one'
)

if val_user_item_feats_df is not None:
    val_user_item_feats_df = val_user_item_feats_df.merge(
        articles,
        left_on='click_article_id',
        right_on='article_id',
        validate='many_to_one'
    )
else:
    val_user_item_feats_df = None

# 当前离线模式没有TestA候选特征
# 不执行原文的tst_user_item_feats_df合并


# ============================================================
# 检查合并结果
# ============================================================

assert len(trn_user_item_feats_df) == trn_rows

if val_user_item_feats_df is not None:
    assert len(val_user_item_feats_df) == val_rows

article_cols = [
    'category_id',
    'created_at_ts',
    'words_count'
]

assert trn_user_item_feats_df[article_cols].notna().all().all()

if val_user_item_feats_df is not None:
    assert val_user_item_feats_df[article_cols].notna().all().all()

print('训练特征:', trn_user_item_feats_df.shape)

if val_user_item_feats_df is not None:
    print('验证特征:', val_user_item_feats_df.shape)

display(trn_user_item_feats_df.head())


训练特征: (301170, 29)
验证特征: (2500000, 29)


,user_id,click_article_id,sim0,time_diff0,word_diff0,sim_max,sim_min,sim_sum,sim_mean,score,...,click_region,click_referrer_type,user_time_hob1,user_time_hob2,cate_list,words_hbo,article_id,category_id,created_at_ts,words_count
0,1,133734,0.113535,26620654000,21,0.113535,0.113535,0.113535,0.113535,0.244705,...,25,6,0.330672,0.993032,[418],176.0,133734,255,1481559255000,155
1,1,31836,0.150834,626430000,1,0.150834,0.150834,0.150834,0.150834,-118.000000,...,25,6,0.330672,0.993032,[418],176.0,31836,26,1507553479000,175
2,2,233717,0.238539,1087514000,21,0.238539,0.238539,0.238539,0.238539,-108.000000,...,25,2,0.330706,0.993024,[43],205.0,233717,375,1507089657000,184
3,3,36162,0.646486,5148000,17,0.646486,0.646486,0.646486,0.646486,0.894016,...,25,2,0.330684,0.993039,[99],188.0,36162,43,1508177171000,205
4,3,540,0.316301,86330217000,17,0.316301,0.316301,0.316301,0.316301,0.276184,...,25,2,0.330684,0.993039,[99],188.0,540,1,1421852102000,171


## 文章的主题是否在用户的爱好里面

In [52]:

# 确认上一节读取的cate_list已恢复为Python列表
assert trn_user_item_feats_df['cate_list'].map(
    lambda x: isinstance(x, list)
).all()

if val_user_item_feats_df is not None:
    assert val_user_item_feats_df['cate_list'].map(
        lambda x: isinstance(x, list)
    ).all()


# 训练数据：判断候选文章类别是否属于用户历史类别
trn_user_item_feats_df['is_cat_hab'] = trn_user_item_feats_df.apply(
    lambda x: 1 if x.category_id in set(x.cate_list) else 0,
    axis=1
)

# 验证数据
if val_user_item_feats_df is not None:
    val_user_item_feats_df['is_cat_hab'] = val_user_item_feats_df.apply(
        lambda x: 1 if x.category_id in set(x.cate_list) else 0,
        axis=1
)
else:
    val_user_item_feats_df = None

# 当前offline=True，不存在TestA候选特征
if tst_user_item_feats_df is not None:
    tst_user_item_feats_df['is_cat_hab'] = tst_user_item_feats_df.apply(
        lambda x: 1 if x.category_id in set(x.cate_list) else 0,
        axis=1
)


# ============================================================
# 删除不再需要的中间字段
# ============================================================

# cate_list已经转换成is_cat_hab，不再需要原始列表
del trn_user_item_feats_df['cate_list']

if val_user_item_feats_df is not None:
    del val_user_item_feats_df['cate_list']

if tst_user_item_feats_df is not None:
    del tst_user_item_feats_df['cate_list']


# article_id与click_article_id表示相同文章，删除冗余列
del trn_user_item_feats_df['article_id']

if val_user_item_feats_df is not None:
    del val_user_item_feats_df['article_id']

if tst_user_item_feats_df is not None:
    del tst_user_item_feats_df['article_id']


# ============================================================
# 检查结果
# ============================================================

print('训练特征:', trn_user_item_feats_df.shape)

if val_user_item_feats_df is not None:
    print('验证特征:', val_user_item_feats_df.shape)

print('\n训练集is_cat_hab分布:')
print(trn_user_item_feats_df['is_cat_hab'].value_counts())

print('\n验证集is_cat_hab分布:')
if val_user_item_feats_df is not None:
    print(val_user_item_feats_df['is_cat_hab'].value_counts())

assert trn_user_item_feats_df['is_cat_hab'].isin([0, 1]).all()

if val_user_item_feats_df is not None:
    assert val_user_item_feats_df['is_cat_hab'].isin([0, 1]).all()

display(trn_user_item_feats_df.head())


训练特征: (301170, 28)
验证特征: (2500000, 28)

训练集is_cat_hab分布:
is_cat_hab
0    218284
1     82886
Name: count, dtype: int64

验证集is_cat_hab分布:
is_cat_hab
0    1888763
1     611237
Name: count, dtype: int64


,user_id,click_article_id,sim0,time_diff0,word_diff0,sim_max,sim_min,sim_sum,sim_mean,score,...,click_country,click_region,click_referrer_type,user_time_hob1,user_time_hob2,words_hbo,category_id,created_at_ts,words_count,is_cat_hab
0,1,133734,0.113535,26620654000,21,0.113535,0.113535,0.113535,0.113535,0.244705,...,1,25,6,0.330672,0.993032,176.0,255,1481559255000,155,0
1,1,31836,0.150834,626430000,1,0.150834,0.150834,0.150834,0.150834,-118.000000,...,1,25,6,0.330672,0.993032,176.0,26,1507553479000,175,0
2,2,233717,0.238539,1087514000,21,0.238539,0.238539,0.238539,0.238539,-108.000000,...,1,25,2,0.330706,0.993024,205.0,375,1507089657000,184,0
3,3,36162,0.646486,5148000,17,0.646486,0.646486,0.646486,0.646486,0.894016,...,1,25,2,0.330684,0.993039,188.0,43,1508177171000,205,0
4,3,540,0.316301,86330217000,17,0.316301,0.316301,0.316301,0.316301,0.276184,...,1,25,2,0.330684,0.993039,188.0,1,1421852102000,171,0


## 保存特征

In [53]:

# ============================================================
# 保存最终特征
# ============================================================

# 检查必要的特征字段
required_cols = [
    'user_id',
    'click_article_id',
    'score',
    'rank',
    'label',
    'is_cat_hab'
]

assert trn_user_item_feats_df[required_cols].notna().all().all()

if val_user_item_feats_df is not None:
    assert val_user_item_feats_df[required_cols].notna().all().all()

    # 训练和验证的特征列应当一致
    assert trn_user_item_feats_df.columns.tolist() == \
           val_user_item_feats_df.columns.tolist()

# 检查标签和候选文章是否存在重复
assert trn_user_item_feats_df['label'].isin([0, 1]).all()
assert not trn_user_item_feats_df.duplicated(
    ['user_id', 'click_article_id']
).any()

if val_user_item_feats_df is not None:
    assert val_user_item_feats_df['label'].isin([0, 1]).all()
    assert not val_user_item_feats_df.duplicated(
        ['user_id', 'click_article_id']
    ).any()


# ============================================================
# 训练、验证及测试特征保存
# ============================================================

os.makedirs(save_path, exist_ok=True)

# 训练验证特征
trn_user_item_feats_df.to_csv(
    save_path + 'trn_user_item_feats_df.csv',
    index=False
)

if val_user_item_feats_df is not None:
    val_user_item_feats_df.to_csv(
        save_path + 'val_user_item_feats_df.csv',
        index=False
    )

# 当前为离线模式，暂不保存TestA特征
if tst_user_item_feats_df is not None:
    tst_user_item_feats_df.to_csv(
        save_path + 'tst_user_item_feats_df.csv',
        index=False
    )


# ============================================================
# 查看最终特征
# ============================================================

print('训练特征:', trn_user_item_feats_df.shape)
print('训练用户数:', trn_user_item_feats_df['user_id'].nunique())
print('训练正样本数:', int(trn_user_item_feats_df['label'].sum()))

if val_user_item_feats_df is not None:
    print('验证特征:', val_user_item_feats_df.shape)
    print('验证候选用户数:', val_user_item_feats_df['user_id'].nunique())
    print('验证正样本数:', int(val_user_item_feats_df['label'].sum()))

print('\n最终特征保存完成')


训练特征: (301170, 28)
训练用户数: 179657
训练正样本数: 99775
验证特征: (2500000, 28)
验证候选用户数: 50000
验证正样本数: 30748

最终特征保存完成
